# CSE 151B — Submission notebook (`private.jsonl`)

Live in **`notebooks/submission.ipynb`**. Same inference stack as [`full_public.ipynb`](full_public.ipynb), run on all of **`data/private.jsonl`** (no labels).

1. (**Colab A100**) `%pip` → restart runtime → Drive copy cell.
2. Set **`MAX_TOKENS`** in §2 (default **32k**, matching pub-003).
3. Prompts are **per question**: baseline for MCQ and single-blank free-form; multi-blank format when `[ANS]` count > 1.
4. Generate with checkpointing → write **`results/submission.csv`**.

**Output:** CSV with columns `id`, `response` (full model trace, CSV-escaped). Checkpoint: `results/submission.responses.jsonl`.

### Google Colab

**Colab (A100):** run the `%pip` cell below, restart runtime, continue. **Local:** use your venv — same packages (`vllm`, `transformers`, …).

> **Note:** `bitsandbytes` is not needed — Qwen3-4B fits in native bf16 on A100 (~8 GB), which is faster than quantized loading.

In [ ]:
# Colab: skip when working locally with an existing venv.
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers

## 2. Imports & configuration

- `MAX_TOKENS` — generation budget (current: 32768)
- `PROMPT_VARIANT` — names the prompt build defined in §5. Baked into the
  output filename so two runs with different prompts cannot share a checkpoint
- `SUBMISSION_CSV` — auto-derived: `results/submission_{PROMPT_VARIANT}_{N}k.csv`
- `DATA_PATH` — `data/private.jsonl`

The responses checkpoint (`{stem}.responses.jsonl`) inherits the same stem, so
changing `PROMPT_VARIANT` automatically gives a fresh checkpoint — no risk of
resuming from responses generated under an older prompt.


In [ ]:
import csv
import json
import os
from pathlib import Path


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()

MAX_TOKENS = 32768

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID   = "0"
DATA_PATH = str(REPO_ROOT / "data/private.jsonl")

# Identifies the prompt build defined in §5. Must match a variant defined in
# notebooks/dev.ipynb when we want dev/submission to agree. Used to namespace
# SUBMISSION_CSV + the responses checkpoint so a prompt change cannot silently
# resume from responses generated under an older prompt.
PROMPT_VARIANT = "precision_xml_verify"

# vLLM's max_model_len is the TOTAL context (prompt + generation). Reserve
# PROMPT_BUDGET for the precision_xml system (~700 tok) + longest expected
# question + chat-template overhead, so changing MAX_TOKENS automatically grows
# max_model_len. Without this, a higher MAX_TOKENS gets silently clamped to
# (max_model_len - prompt_len).
PROMPT_BUDGET = 2048
MAX_MODEL_LEN = MAX_TOKENS + PROMPT_BUDGET

_TOKEN_K = MAX_TOKENS // 1024
SUBMISSION_CSV = str(REPO_ROOT / f"results/submission_{PROMPT_VARIANT}_{_TOKEN_K}k.csv")

print(f"REPO_ROOT      = {REPO_ROOT}")
print(f"MAX_TOKENS     = {MAX_TOKENS} ({_TOKEN_K}k)   # new-token cap")
print(f"MAX_MODEL_LEN  = {MAX_MODEL_LEN}             # prompt + generation cap")
print(f"PROMPT_VARIANT = {PROMPT_VARIANT}")
print(f"SUBMISSION_CSV = {SUBMISSION_CSV}")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import contextlib
import io
import sys
from typing import Optional

@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close()
            sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Colab: copy `private.jsonl` from Drive

Upload **`private.jsonl`** to `My Drive/CSE151B/data/private.jsonl` (or change `DRIVE_PRIVATE`). Skip on local clone when `data/private.jsonl` exists.

In [ ]:
import shutil
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — use repo `data/private.jsonl`.")
else:
    drive.mount("/content/drive")
    DRIVE_PRIVATE = Path("/content/drive/MyDrive/CSE151B/data/private.jsonl")
    DRIVE_PROJECT_ROOT = DRIVE_PRIVATE.parent.parent
    if not DRIVE_PRIVATE.is_file():
        raise FileNotFoundError(
            f"Missing on Drive: {DRIVE_PRIVATE}\nUpload private.jsonl or set DRIVE_PRIVATE."
        )
    (REPO_ROOT / "data").mkdir(parents=True, exist_ok=True)
    dest = REPO_ROOT / "data/private.jsonl"
    shutil.copy2(DRIVE_PRIVATE, dest)
    print(f"Copied to {dest.resolve()}")

## 4. Load dataset

Private rows have `question`, optional `options`, and `id` — **no** `answer` field.

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |

In [ ]:
def n_ans_placeholders(question: str) -> int:
    return question.count("[ANS]")


data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form) from {DATA_PATH}")

mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))
multi_sample = next(
    (d for d in data if not d.get("options") and n_ans_placeholders(d["question"]) > 1),
    free_sample,
)

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2)[:1200], "...\n" if len(json.dumps(mcq_sample)) > 1200 else "\n")
print("── Free-form sample ──")
print(json.dumps(free_sample, indent=2)[:1200], "...\n" if len(json.dumps(free_sample)) > 1200 else "\n")
if multi_sample is not free_sample:
    print(f"── Multi-blank sample ({n_ans_placeholders(multi_sample['question'])} blanks) ──")
    print(json.dumps(multi_sample, indent=2)[:1200], "...\n" if len(json.dumps(multi_sample)) > 1200 else "\n")

## 5. Prompt construction

Variant: **`precision_xml_verify`** (mirrors `notebooks/dev.ipynb`). Same system+user XML scaffolding for every question; the user-side body switches on question type.

**System prompt** (math and MCQ): `<instructions>` carries role + brief self-verify + concise directive as a single procedural sentence (NOT a `<rule>` tag — earlier `<rule>` placement caused 24% of responses to echo the directive back into output). `<response_format>` then lists declarative grader-aware rules (exact-vs-decimal, box hygiene, multi-select, single Final Answer line, no-duplicate-box, separator, multi-blank layout).

**User prompt** per question type:

| Case | User body |
|------|-----------|
| MCQ                       | `<problem><statement>…</statement><options><option letter="A">…</option>…</options><output_format>one \boxed{letter}</output_format></problem>` |
| Free-form, 1 `[ANS]`      | `<problem><statement>…</statement><output_format>one \boxed{value}</output_format></problem>` |
| Free-form, 2+ `[ANS]`     | `<problem><statement>…</statement><blanks count="N"/><output_format>N comma-separated \boxed{…} values, in `[ANS]` order</output_format></problem>` |

Tags are plain text inside the Qwen3 chat template — not special tokens. The model attends to them as structural anchors learned from pretraining.


In [ ]:
# ── precision_xml_verify (mirrors notebooks/dev.ipynb precision_xml_verify) ───
# System: <instructions> carries role + verify + concise as a single procedural
# directive (NOT a <rule>, to avoid the rule-echo pathology seen in dev-xml-001);
# <response_format> stays declarative.
# User:   <problem><statement>…</statement>[<blanks count> | <options>…]
#         <output_format>…</output_format></problem>
# Independent of any wrapper flag — emits this structure unconditionally.

_ROLE_SOLVE_RIGOROUS = (
    "You are a precise mathematical reasoner. Solve the problem rigorously "
    "and keep your reasoning concise — do not re-derive steps you have already "
    "completed or go in circles. After your reasoning, briefly check your "
    "final answer for errors or contradictions (substitute back, verify "
    "units/range/signs, or confirm with an alternative method) before "
    "writing it inside \\boxed{}."
)

_ROLE_MCQ_RIGOROUS = (
    "You are a precise mathematical reasoner. Read the problem and the answer "
    "choices below, then rigorously identify the single best answer. Keep your "
    "reasoning concise. Before committing, briefly verify by elimination or "
    "by substitution into the constraints."
)

# ── grader-aware response_format rules (math) ────────────────────────────────
_PRECISION_V2_CLAUSE = (
    "For pure algebra, geometry, trig identities, and combinatorics, prefer EXACT form "
    "(fractions, radicals, \\pi). For statistics, probability, hypothesis tests, "
    "confidence intervals, finance, physics, numerical methods, and any question that "
    "says 'estimate' or 'approximate', use DECIMAL form with at least 10 significant "
    "figures of intermediate precision, and report at least 10 significant figures in "
    "the box without rounding. When in doubt, prefer decimal — the grader's numeric "
    "path is more reliable than its symbolic path."
)
_GRADER_FORMAT_CLAUSE = (
    "Box the value only — no units or words (write \\boxed{5}, not \\boxed{5 cm}). "
    "Do not use a percent sign: give a probability as a decimal (e.g. 0.5) and a "
    "percentage as a bare number (e.g. 50). Keep a coordinate, tuple, or interval "
    "together in ONE box with its brackets, e.g. \\boxed{(-1,-3)} — never split it "
    "across boxes. For yes/no answers box just \\boxed{Yes} or \\boxed{No}."
)
_MULTI_SELECT_CLAUSE = (
    "For a slot that asks 'select all that apply' or expects multiple option letters, "
    "concatenate the letters in alphabetical order inside ONE box with no separator: "
    "\\boxed{AB} — never \\boxed{A,B}, and never two boxes."
)

_MATH_RESPONSE_RULES = [
    _PRECISION_V2_CLAUSE,
    _GRADER_FORMAT_CLAUSE,
    _MULTI_SELECT_CLAUSE,
    "Put your \\boxed{} answers ONLY in a single Final Answer line at the very "
    "end. Do not put \\boxed{} in bullets, headers, sub-conclusions, or "
    "'Answer:' lines above that final line, and do not restate the boxed "
    "answers elsewhere.",
    "Emit each final \\boxed{} EXACTLY ONCE — do not repeat the same boxed "
    "value for emphasis (e.g. never write \\boxed{G} \\boxed{G}). The "
    "grader's contiguous-group rule joins adjacent duplicate boxes into 'G, G' "
    "and rejects the answer.",
    "Between final boxes use exactly a comma and a space ', '. Never put "
    "\\quad, \\qquad, \\;, \\,, \\hspace, &, or any LaTeX spacing macro "
    "between boxes (the grader's contiguous-group rule drops boxes after such "
    "a separator).",
    "For problems with multiple [ANS] placeholders, emit one \\boxed{} per "
    "blank in the order the blanks appear (e.g. \\boxed{3}, \\boxed{7}). "
    "Do not use labels like 'Answer 1:' between boxes.",
    "For single-answer problems, use exactly one \\boxed{}.",
]

_MCQ_RESPONSE_RULES = [
    "End with a single Final Answer line containing exactly one \\boxed{} "
    "that wraps ONLY the letter of your chosen option (e.g. \\boxed{C}). "
    "Emit this box EXACTLY ONCE — never repeat it for emphasis "
    "(e.g. never write \\boxed{C} \\boxed{C}).",
    "If the slot expects multiple letters ('select all that apply'), "
    "concatenate them alphabetically inside one box with no separator "
    "(e.g. \\boxed{AB}).",
]


def _xml_rule_block(tag: str, rules: list) -> str:
    inner = "\n".join(f"<rule>{r}</rule>" for r in rules)
    return f"<{tag}>\n{inner}\n</{tag}>"


SYSTEM_PROMPT_MATH = (
    f"<instructions>{_ROLE_SOLVE_RIGOROUS}</instructions>\n\n"
    f"{_xml_rule_block('response_format', _MATH_RESPONSE_RULES)}"
)

SYSTEM_PROMPT_MCQ = (
    f"<instructions>{_ROLE_MCQ_RIGOROUS}</instructions>\n\n"
    f"{_xml_rule_block('response_format', _MCQ_RESPONSE_RULES)}"
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt). Structure mirrors dev's _build_precision_xml_user."""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opt_tags = "\n".join(
            f'  <option letter="{lbl}">{opt.strip()}</option>'
            for lbl, opt in zip(labels, options)
        )
        user = (
            "<problem>\n"
            f"  <statement>{question}</statement>\n"
            f"  <options>\n{opt_tags}\n  </options>\n"
            "  <output_format>End with a single Final Answer line: one "
            "\\boxed{} containing only the chosen option letter.</output_format>\n"
            "</problem>"
        )
        return SYSTEM_PROMPT_MCQ, user

    n_blanks = n_ans_placeholders(question)
    if n_blanks > 1:
        example = ", ".join("\\boxed{...}" for _ in range(n_blanks))
        out_fmt = (
            f"End with a single Final Answer line containing {n_blanks} "
            f"comma-separated \\boxed{{}} values in the order the [ANS] "
            f"blanks appear: {example}."
        )
        blanks_tag = f'  <blanks count="{n_blanks}"/>\n'
    else:
        out_fmt = "End with a single Final Answer line: one \\boxed{} value."
        blanks_tag = ""
    user = (
        "<problem>\n"
        f"  <statement>{question}</statement>\n"
        f"{blanks_tag}"
        f"  <output_format>{out_fmt}</output_format>\n"
        "</problem>"
    )
    return SYSTEM_PROMPT_MATH, user


def prompt_mode(question: str, options: Optional[list]) -> str:
    if options:
        return "mcq/precision_xml_verify"
    return "multi-blank/precision_xml_verify" if n_ans_placeholders(question) > 1 else "single-blank/precision_xml_verify"


print(f"Active variant: precision_xml_verify (verify in <instructions>)")
print(f"\nMath system prompt (first 240 chars):\n  {SYSTEM_PROMPT_MATH[:240]}...")
print(f"\nMCQ  system prompt (first 240 chars):\n  {SYSTEM_PROMPT_MCQ[:240]}...\n")

for label, item in [
    ("MCQ", mcq_sample),
    ("Free-form (single-blank)", free_sample),
    *( [(f"Multi-blank ({n_ans_placeholders(multi_sample['question'])} blanks)", multi_sample)] if multi_sample is not free_sample else [] ),
]:
    mode = prompt_mode(item["question"], item.get("options"))
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} [{mode}] ──")
    print(f"  user (first 300 chars): {usr_p[:300]}...\n")


## 6. Load model with vLLM (A100 profile)

Same **bf16** profile as `full_public.ipynb` — not the starter L4 INT8 path. See [`docs/infra/vllm-inference-config.md`](../docs/infra/vllm-inference-config.md).

| Parameter | Value | Notes |
|-----------|-------|-------|
| `dtype` | `bfloat16` | |
| `max_model_len` | `MAX_TOKENS + PROMPT_BUDGET` | derived in §2; total prompt+generation context |
| `max_num_seqs` | auto-scaled from `max_model_len` | halved as context grows so KV cache fits |
| `max_num_batched_tokens` | `max(max_model_len, 32768)` | prefill batch size |
| `enable_prefix_caching` | True | shared system prompt cached across items |
| `enable_chunked_prefill` | True | |
| `kv_cache_dtype` | `fp8` | |

To change generation budget: edit `MAX_TOKENS` in §2 only — `max_model_len`, `max_num_seqs`, and the output filename all follow.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# max_num_seqs heuristic: KV cache per seq grows ~linearly with max_model_len.
# At 17500 the A100 profile ran 256 seqs comfortably; halve for ~34k context.
_max_num_seqs = max(64, int(256 * 17500 / MAX_MODEL_LEN))

with _jupyter_stdout_for_vllm():
    llm = LLM(
        model=MODEL_ID,
        dtype="bfloat16",
        enable_prefix_caching=True,
        gpu_memory_utilization=0.88,
        max_model_len=MAX_MODEL_LEN,
        trust_remote_code=True,
        max_num_seqs=_max_num_seqs,
        max_num_batched_tokens=max(MAX_MODEL_LEN, 32768),
        enable_chunked_prefill=True,
        kv_cache_dtype="fp8",
    )

default_sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print(f"Model loaded.  max_model_len={MAX_MODEL_LEN}  max_num_seqs={_max_num_seqs}  sampling.max_tokens={MAX_TOKENS}")


## 7. Generate responses

Checkpoint: `results/submission.responses.jsonl` (Drive on Colab). Delete checkpoint to regenerate from scratch.

In [ ]:
CHUNK_SIZE = 128

try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(SUBMISSION_CSV).parent

_results_dir.mkdir(parents=True, exist_ok=True)
response_checkpoint = _results_dir / (Path(SUBMISSION_CSV).stem + ".responses.jsonl")
print(f"Checkpoint path: {response_checkpoint}")

completed: dict = {}
if response_checkpoint.exists():
    with open(response_checkpoint) as f:
        for line in f:
            r = json.loads(line)
            completed[r["id"]] = r["response"]
    print(f"Resumed from checkpoint: {len(completed)} responses already generated")

remaining = [d for d in data if d.get("id") not in completed]
print(f"Generating {len(remaining)} remaining / {len(data)} total...")

for chunk_start in range(0, len(remaining), CHUNK_SIZE):
    chunk = remaining[chunk_start : chunk_start + CHUNK_SIZE]

    prompts = []
    chunk_params = []
    for item in chunk:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)
        chunk_params.append(default_sampling_params)

    with _jupyter_stdout_for_vllm():
        outputs = llm.generate(prompts, sampling_params=chunk_params)

    with open(response_checkpoint, "a") as f:
        for item, out in zip(chunk, outputs):
            response = out.outputs[0].text.strip()
            completed[item["id"]] = response
            f.write(json.dumps({"id": item["id"], "response": response}) + "\n")

    done = len(completed)
    print(f"  {done}/{len(data)}  ({done / len(data) * 100:.1f}%)")

assert len(completed) == len(data), "Missing ids — checkpoint vs DATA_PATH mismatch"
responses = [completed[d["id"]] for d in data]
print(f"Done. {len(responses)} responses ready.")

## 8. Write submission CSV

Uses Python’s `csv` writer so commas and newlines inside `response` are quoted per RFC 4180.

In [ ]:
try:
    csv_out = DRIVE_PROJECT_ROOT / "results" / Path(SUBMISSION_CSV).name
except NameError:
    csv_out = Path(SUBMISSION_CSV)

csv_out.parent.mkdir(parents=True, exist_ok=True)

with open(csv_out, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
    w.writerow(["id", "response"])
    for row in data:
        qid = row["id"]
        w.writerow([qid, completed[qid]])

print(f"Wrote {len(data)} rows to {csv_out.resolve()}")

with open(csv_out, newline="", encoding="utf-8") as f:
    n = sum(1 for _ in csv.reader(f))
assert n == len(data) + 1, f"Expected header + {len(data)} rows, got {n} lines"
print("CSV line count OK (1 header +", len(data), "data rows).")

## Done

Upload **`submission.csv`** to the course leaderboard workflow. Keep **`submission.responses.jsonl`** as a backup of raw traces.

Pipeline matches **pub-003** (`full_public.ipynb`): 32k tokens, bf16 A100, adaptive multi-blank prompts.

In [ ]:
try:
    from google.colab import runtime
    drive.flush_and_unmount()
    runtime.unassign()
except ImportError:
    print("Not running in Colab — skipping runtime termination.")
except NameError:
    print("Drive not mounted — skipping runtime termination.")